# Exploración

In [ ]:
from LanguageDatasets import LanguageDataset

## Asturiano

In [3]:
ast = LanguageDataset("asturiano",True)

len(ast.json)

Descargando tatoeba para asturiano:
Completado con éxito


448

In [3]:
len(ast["nlbb"])

2721557

In [4]:
ast.raw_datasets

{'tatoeba': {'start': 0, 'end': 463}, 'nlbb': {'start': 463, 'end': 2722020}}

In [5]:
ast[-20:]

[{'text': 'La compra realizada en sema.com.do también puede ser retirada en una de nuestras sucursales en un plazo menor a horas laborables.'},
 {'text': 'Lorient Capital Intercéltica capital apaez na entrada de la ciudá Festival Intercéltico de Lorient'},
 {'text': 'Marcelino Camacho, l históricu líder de Comisiones Obreras, morriera esta madrugada pasada.'},
 {'text': 'Matthew : And whatsoeuer ye shall aske in prayer, if ye beleeue, ye shall receiue it.'},
 {'text': 'Matthew : O ye serpentes, O ye generacion of vypers, how wyl ye escape the damnacion of Hell?'},
 {'text': 'Numbers : And Aaron dyd so, set ye lampes vpo ye candilsticke, as ye LORDE comaunded Moses.'},
 {'text': 'Para cerrar el pedido ponte antes en contacto escribiendo a gema escritasamano.com, ya que necesitaremos la lista de invitados así como la mesa asignada a cada uno.'},
 {'text': 'Piscinas Pañales Piscinas Pañales Pañales Piscinas Para Piscinas Pañales Para Para Para Pañales H IWE D'},
 {'text': 'Por Vestidos La

nlbb: {'encoding': 'utf-8', 'confidence': 0.99, 'language': ''}

In [6]:
def tail(filename, n=20, encoding="utf-8"):
    with open(filename, "r", encoding=encoding) as f:
        lines = f.readlines()
    return lines[-n:]

# Ejemplo
ultimas = tail("datasets/asturiano/nlbb.txt", 20)
for linea in ultimas:
    print(linea.strip())


Para cerrar el pedido ponte antes en contacto escribiendo a gema@escritasamano.com, ya que necesitaremos la lista de invitados así como la mesa asignada a cada uno.
﻿Piscinas Pañales Piscinas Pañales Pañales Piscinas Para Piscinas Pañales Para Para Para Pañales H2IWE9D
﻿Por Vestidos La Por Kaos Rodilla Kaos Rodilla Rodilla Kaos La Vestidos La Por Vestidos Ix1RqwPYRv
﻿Proverbs 15:25 The LORDE wyl breake downe ye house of ye proude, but he shal make fast ye borders of ye wyddowe.
﻿Psalms 135 O prayse ye name of ye LORDE, praise it o ye seruautes of ye LORDE.
﻿Psalms 135:1 O prayse ye name of ye LORDE, praise it o ye seruautes of ye LORDE.
﻿Reservados todos los derechos de autor: cctvradio.com | Dirección: Calle La Vendimia Mz E Lote 1 Urb.
﻿Servicio telefónico para el Ciudadano, S.L. inscrita en el Registro Mercantil de Huesca, Tomo 523, Folio 141, hoja HU-11201, inscripción 1ª con domicilio en C/Ainielle 24, 22005 Huesca, con CIF B22374896
﻿Tiendas SanxenxoOfertasFolletos Y Tiendas Sanx

# Generando dataset Ortográficos

In [ ]:
import numpy as np
import pandas as pd
from generateDataset import generateDatasetOrtograficoAnotado, quitarYaAnotadas
from LanguageDatasets import LanguageDataset
import os
from dotenv import load_dotenv
load_dotenv("secrets.env")

C:\Users\migue\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## Gallego

In [ ]:
gl = LanguageDataset("gallego")
gl.read_local_file("EvalDatasets/AnotadoRaw","gallego.csv")
len(gl.json)

700

## Aranés

In [ ]:
oc = LanguageDataset("aranes")
oc.read_local_file("EvalDatasets/AnotadoRaw","aranes.csv")
len(oc.json)

705

## Asturiano

In [ ]:
ast = LanguageDataset("asturiano")
ast.read_local_file("EvalDatasets/AnotadoRaw","asturiano.csv")
len(ast.json)

701

## Generar datasets

In [ ]:
for lang, dataset in [("asturiano", ast), ("aranes", oc), ("gallego", gl)]:
    print(f"Empezando {lang}...")
    csv_path = f"EvalDatasets/Anotado/{lang}.csv"

    # Si no existe, crear CSV vacío con columnas correctas
    if not os.path.exists(csv_path):
        pd.DataFrame(columns=["original", "annotated", "n_errors"]).to_csv(csv_path, index=False)
        print(f"Creando nuevo para el {lang}: {csv_path}")
    else:
        # Cargar anotadas
        anotadas = pd.read_csv(csv_path)
        # Filtrar dataset
        dataset = quitarYaAnotadas(dataset, anotadas)

    # Generar nuevas anotaciones
    evalDataset = generateDatasetOrtograficoAnotado(
        dataset,
        os.getenv("GROQ_API_KEY"),
        save=False,
        model="openai/gpt-oss-120b"
    )

    # Guardar concatenado
    df_final = pd.concat([pd.read_csv(csv_path), evalDataset], ignore_index=True)
    df_final.to_csv(csv_path, index=False)
    print(f"Dataset anotado en {lang} creado con éxito en {csv_path}.\n")
print("Filas nuevas generadas:", evalDataset.shape[0])


Empezando asturiano...

Dataset Generado
Dataset anotado en asturiano creado con éxito en Anotado/asturiano.csv.

Empezando aranes...

Dataset Generado
Dataset anotado en aranes creado con éxito en Anotado/aranes.csv.

Empezando gallego...
Progreso: 100.0% (133/133) | step: 3 s, remaining time: 0 min 0 ss
Dataset Generado
Dataset anotado en gallego creado con éxito en Anotado/gallego.csv.

Filas nuevas generadas: 133


# Generación dataset con huecos

In [1]:
import numpy as np
import pandas as pd
from generateDataset import generateDatasetHuecos
from LanguageDatasets import LanguageDataset
import os
from dotenv import load_dotenv
load_dotenv("secrets.env")

C:\Users\migue\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
with open("Evaldatasets/Raw/idioms_train_es.txt", "r", encoding="utf-8") as fEsp:
    esp = fEsp.readlines()

with open("Evaldatasets/Raw/idioms_train_gl.txt", "r", encoding="utf-8") as fGl:
    gl = fGl.readlines()

with open("Evaldatasets/Raw/idioms_test_es.txt", "r", encoding="utf-8") as fEsp:
    espTest = fEsp.readlines()

with open("Evaldatasets/Raw/idioms_test_gl.txt", "r", encoding="utf-8") as fGl:
    glTest = fGl.readlines()


gallego = pd.DataFrame(np.array((esp + espTest, gl + glTest)).T, columns=["es","gl"])
gallego["text"] = gallego["gl"]
gl = LanguageDataset("gallego")
gl.read_dataframe(gallego[["es","text"]])
gl.json = gl[:700]
generateDatasetHuecos(gl,directory="EvalDatasets/Huecos")

,original,masked_sentence,missing_word
0,Aquela arroutada converteuse nun movemento de ...,Aquela arroutada converteuse nun movemento de ...,cambiou
1,"A realidade, por dura que sexa, sempre nos lem...","A realidade, por dura que sexa, <mask> nos lem...",sempre
2,"A miña avoa espichou onte, mais deixounos cheo...","A miña avoa <mask> onte, mais deixounos cheos ...",espichou
3,O cantante limpou a gorxa facendo ademán de co...,<mask> cantante limpou a gorxa facendo ademán ...,O
4,"O home, ao ver a súa exmuller no bar, pagou a ...","O home, ao ver a súa <mask> no bar, pagou a co...",exmuller
...,...,...,...
695,A túa capacidade para analizar situacións comp...,A túa capacidade para analizar situacións comp...,a
696,Alá vai a festa! Estaba convencido de que me í...,Alá vai a festa! Estaba convencido de que me í...,admitir
697,O neno colleu a pelota facendo ademán de lanza...,O neno <mask> a pelota facendo ademán de lanza...,colleu
698,"O seu amigo de toda a vida, por un malentendid...","O seu amigo de <mask> a vida, por un malentend...",toda


In [6]:
# Login using e.g. `huggingface-cli login` to access this dataset
aranes = pd.read_parquet("hf://datasets/projecte-aina/ES-OC_Parallel_Corpus/es-arn_corpus.parquet").sample(7000)
aranes["text"] = aranes["arn"]
ar = LanguageDataset("aranes")
ar.read_dataframe(aranes[["es","text"]])
ar.json = ar[:700]
generateDatasetHuecos(ar,directory="EvalDatasets/Huecos")

,original,masked_sentence,missing_word
0,Moscòu ei era mair de totes es ciutats.,Moscòu ei <mask> mair de totes es ciutats.,era
1,Candidatures ath Conselh Generau d'Aran,Candidatures ath Conselh <mask> d'Aran,Generau
2,Que non me mingi ara gent.,Que <mask> me mingi ara gent.,non
3,"Se vòs, te podem amiar damb nosati e presentar...",Se <mask> te podem amiar damb nosati e present...,"vòs,"
4,Prevencion Pesta Porcina Africana PPA : D'acòr...,Prevencion <mask> Porcina Africana PPA : D'acò...,Pesta
...,...,...,...
695,f Establir eth catalòg de drets des hemnes que...,<mask> Establir eth catalòg de drets des hemne...,f
696,Per mès qu'era comdessa amaguèsse es sues inte...,Per mès qu'era comdessa amaguèsse es sues inte...,pensaue
697,"E atau li cau èster, atau li cau èster!","E atau li cau èster, atau <mask> cau èster!",li
698,Tanben se dissòlv era camèra de forma automati...,Tanben se dissòlv era camèra de forma automati...,dempús


In [5]:
# Login using e.g. `huggingface-cli login` to access this dataset
asturiano = pd.read_parquet("hf://datasets/projecte-aina/ES-AST_Parallel_Corpus/es-ast_corpus.parquet").sample(7000)
asturiano["text"] = asturiano["ast"]
ast = LanguageDataset("asturiano")
ast.read_dataframe(asturiano[["es","text"]])
ast.json = ar[:700]
generateDatasetHuecos(ast,directory="EvalDatasets/Huecos")

,original,masked_sentence,missing_word
0,"Eth plan aprovat provisionaument, en complimen...","Eth plan aprovat provisionaument, en complimen...",Departament
1,È d'èster aquiu entà ensenhar-te es tonos de g...,È d'èster aquiu entà <mask> es tonos de gris.,ensenhar-te
2,Quan er organ que hè era proposicion non segui...,Quan <mask> organ que hè era proposicion non s...,er
3,Aquera qu'ère era dusau mòrt que jo vedia e et...,<mask> qu'ère era dusau mòrt que jo vedia e et...,Aquera
4,"E dempús dera dança, cada ua d’eres lancèc dèt...","E dempús dera dança, cada ua d’eres lancèc <ma...",dètz
...,...,...,...
695,"Que non ei repotegable evitar un malastre, enc...","Que non ei repotegable <mask> un malastre, enc...",evitar
696,Guaires beròies e blanques hemnes sauvères pet...,Guaires beròies e blanques hemnes sauvères pet...,"desorde,"
697,A Ulisses li mestrèren es que se tenguien ad a...,A <mask> li mestrèren es que se tenguien ad aq...,Ulisses
698,Se produsiren controvèrsies pes conflictes pol...,Se produsiren controvèrsies pes conflictes pol...,economica
